# Cobaya + hmfast: full-sky binned $D_\ell$ (tSZ-only, no foreground)

This notebook shows how to:

1. Use **binned $D_\ell$** mock data and a **full-sky covariance** (`covmat_Dl_binned_fullsky_signal_only.npy`) from `synthetic_data`.
2. Drive inference with **Cobaya**, while theory uses **hmfast** (JAX / optional **GPU**) via the helper module `hmfast_cobaya_theory_fullsky.py`.
3. **Foreground**: omitted — likelihood matches **signal-only** data (same spirit as `tszpower_scatter_fullsky_signal_only.yaml`).
4. **Pressure-profile nuisance parameters** (`A_\mathrm{SZ}`, `alpha_\mathrm{SZ}`, `sigma_{\ln Y}`): **not** sampled — hmfast uses its built-in **GNFW** normalization (different from the tszsbi parametric amplitude).

**Important:** The synthetic vector was produced with the **tszsbi parametric** pipeline, not hmfast. Expect a **poor absolute $\chi^2$ fit** unless you regenerate mocks with hmfast; this tutorial is about **wiring** Cobaya, data, and hmfast.

**Requirements:** `cobaya`, `getdist` (optional), `pyyaml`, `hmfast` installed (e.g. `pip install -e .` from the repo root), JAX, and the tszsbi path is **not** required for the minimal pipeline (only file paths under `tsz_cnc_scatter/synthetic_data`).

## 1. Paths and JAX device (notebook process)

Set **`TUTORIAL_DIR`** to this folder and **`HMFAST_SRC`** to `.../hmfast/src`. The environment variable **`HMFAST_COBAYA_USE_GPU`** is read when `hmfast_cobaya_theory_fullsky` is first imported (set it the same way in the shell before `cobaya-run`).

Run the notebook with the working directory set to `tutorial/` (or adjust paths).

In [1]:
import os
from pathlib import Path

# --- Edit if your clone lives elsewhere ---
TUTORIAL_DIR = Path("/home/lxu/scratch/compute_packages/hmfast/tutorial").resolve()
HMFAST_SRC = TUTORIAL_DIR.parent / "src"
DATA_DIR = Path("/scratch/scratch-lxu/tsz_cnc_scatter/synthetic_data")

USE_GPU = True  # notebook-side JAX; mirror in subprocess via HMFAST_COBAYA_USE_GPU
if not USE_GPU:
    os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["HMFAST_COBAYA_USE_GPU"] = "1" if USE_GPU else "0"

if str(HMFAST_SRC) not in os.environ.get("PYTHONPATH", ""):
    os.environ["PYTHONPATH"] = f"{HMFAST_SRC}:{TUTORIAL_DIR}:" + os.environ.get(
        "PYTHONPATH", ""
    )

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
print("JAX devices:", jax.devices())
print("HMFAST_COBAYA_USE_GPU =", os.environ.get("HMFAST_COBAYA_USE_GPU"))

/scratch/scratch-lxu/venv/cmbagent_env/lib/python3.12/site-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda13_plugin version 0.8.2 is installed, but it is not compatible with the installed jaxlib version 0.10.0, so it will not be used.
  warnings.warn(


JAX devices: [CudaDevice(id=0), CudaDevice(id=1)]
HMFAST_COBAYA_USE_GPU = 1


## 2. Data vector and covariance (binned $D_\ell$)

- **Data:** `Dl_binned_fullsky_signal_only_real0.txt` — columns `ell_eff`, `D_binned`.
- **Covariance:** `covmat_Dl_binned_fullsky_signal_only.npy` (NumPy array; `.txt` also exists and is equivalent).

The custom likelihood `tSZ_PS_Likelihood_FullSkyNPY` in `cobaya_tutorial_likelihood.py` loads `.npy` covariances.

In [2]:
import numpy as np

data_path = DATA_DIR / "Dl_binned_fullsky_signal_only_real0.txt"
cov_npy = DATA_DIR / "covmat_Dl_binned_fullsky_signal_only.npy"

D = np.loadtxt(data_path)
ell, Dl = D[:, 0], D[:, 1]
C = np.load(cov_npy)
print("N_bins =", len(ell), "ell range", ell[0], "..", ell[-1])
print("cov shape", C.shape, "symmetric?", np.allclose(C, C.T))

N_bins = 18 ell range 10.0 .. 959.5
cov shape (18, 18) symmetric? True


## 3. In-process check: one likelihood evaluation (Cobaya `get_model`)

This imports `hmfast_cobaya_theory_fullsky` **after** the env block above, so GPU/CPU matches your choice. Uses the same `cobaya_tszhmfast_fullsky.yaml` blocks loaded via `yaml`.

In [3]:
import yaml
from cobaya.model import get_model

yaml_path = TUTORIAL_DIR / "cobaya_tszhmfast_fullsky.yaml"
with open(yaml_path, "r") as f:
    info = yaml.safe_load(f)

# Point files to DATA_DIR (override yaml scratch paths if needed)
info["likelihood"]["cobaya_tutorial_likelihood.tSZ_PS_Likelihood_FullSkyNPY"]["data_directory"] = str(
    DATA_DIR
)
info["theory"]["hmfast_cobaya_theory_fullsky.HMFastTSZFullSky"][
    "data_vector_file"
] = str(data_path)
info["output"] = str(TUTORIAL_DIR / "chains" / "notebook_smoke")

model = get_model(info)
point = {
    "H0": 67.4,
    "ln10_10A_s": 2.9718,
    "n_s": 0.962,
    "omega_b": 0.022,
    "Omega_m": 0.3096,
}
# Cobaya returns derived params as a list ordered like parameterization.derived_params()
loglike_parts, derived_list = model.loglikes(point)
derived_names = list(model.parameterization.derived_params())
derived = dict(zip(derived_names, derived_list))
print("per-likelihood loglikes", loglike_parts)
print("omega_cdm", derived.get("omega_cdm"))

[model] *WARNING* Ignored blocks/options: ['output', 'sampler', 'logging_level']


Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified



[hmfast_cobaya_theory_fullsky.hmfasttszfullsky] HMFastTSZFullSky: 18 ell bins, ell in [10.0, 959.5]
[cobaya_tutorial_likelihood.tsz_ps_likelihood_fullskynpy] tSZ_PS_Likelihood_FullSkyNPY: loaded 18 bins from Dl_binned_fullsky_signal_only_real0.txt; cov from covmat_Dl_binned_fullsky_signal_only.npy
per-likelihood loglikes [-1165.02342181]
omega_cdm 0.11864384960000002


## 4. Run Cobaya in a **subprocess** (`cobaya-run`)

MCMC uses JAX inside the worker; set **`HMFAST_COBAYA_USE_GPU=1`** (default) or **`0`** for CPU-only in the **child** environment. Extend `PYTHONPATH` with **`hmfast/src`** and this **`tutorial/`** directory so Cobaya can import `hmfast_cobaya_theory_fullsky` and `cobaya_tutorial_likelihood`.

Uncomment and run when you are ready for a real chain (can be slow with hmfast per-step).

In [ ]:
import subprocess
import shutil

def run_cobaya_subprocess(use_gpu: bool = True, dry_run: bool = True):
    exe = shutil.which("cobaya-run")
    if not exe:
        raise RuntimeError("cobaya-run not found on PATH; activate the env with Cobaya installed.")
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{HMFAST_SRC}:{TUTORIAL_DIR}:{env.get('PYTHONPATH','')}"
    env["HMFAST_COBAYA_USE_GPU"] = "1" if use_gpu else "0"
    cfg = TUTORIAL_DIR / "cobaya_tszhmfast_fullsky.yaml"
    cmd = [exe, str(cfg)]
    print("CMD:", " ".join(cmd))
    print("HMFAST_COBAYA_USE_GPU =", env["HMFAST_COBAYA_USE_GPU"])
    if dry_run:
        print("Set dry_run=False to execute.")
        return None
    return subprocess.run(cmd, cwd=str(TUTORIAL_DIR), env=env, check=False)

# Example (does not start MCMC until you set dry_run=False):
run_cobaya_subprocess(use_gpu=USE_GPU, dry_run=False)

CMD: /scratch/scratch-lxu/venv/cmbagent_env/bin/cobaya-run /scratch/scratch-lxu/compute_packages/hmfast/tutorial/cobaya_tszhmfast_fullsky.yaml
HMFAST_COBAYA_USE_GPU = 1


Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified



[output] Output to be read-from/written-into folder 'chains/tutorial_hmfast_tszhmfast_fullsky', with prefix 'chains'


/scratch/scratch-lxu/venv/cmbagent_env/lib/python3.12/site-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda13_plugin version 0.8.2 is installed, but it is not compatible with the installed jaxlib version 0.10.0, so it will not be used.
  warnings.warn(


[jax._src.xla_bridge] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
[hmfast_cobaya_theory_fullsky.hmfasttszfullsky] HMFastTSZFullSky: 18 ell bins, ell in [10.0, 959.5]
[cobaya_tutorial_likelihood.tsz_ps_likelihood_fullskynpy] tSZ_PS_Likelihood_FullSkyNPY: loaded 18 bins from Dl_binned_fullsky_signal_only_real0.txt; cov from covmat_Dl_binned_fullsky_signal_only.npy
[mcmc] Getting initial point... (this may take a few seconds)
[mcmc] Initial point: H0:66.05186, ln10_10A_s:3.041702, n_s:0.9637027, omega_b:0.02276325, Omega_m:0.2918913
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {cobaya_tutorial_likelihood.tSZ_PS_Likelihood_FullSkyNPY: 21200.0, hmfast_cobaya_theory_fullsky.HMFastTSZFullSky: 0.195}
[mcmc] Covariance matrix not present. We will start learning the covariance of the proposal earlier: R-1 = 30 (would be 2 if all params loaded).


## 5. Reference YAML layout

File: **`cobaya_tszhmfast_fullsky.yaml`**. Priors match the spirit of `tszpower_scatter_fullsky_signal_only.yaml` for cosmological parameters only; **no** `A_SZ` / `alpha_SZ` / `sigma_lnY` block.

| Piece | Module |
|-------|--------|
| Theory | `hmfast_cobaya_theory_fullsky.HMFastTSZFullSky` |
| Likelihood | `cobaya_tutorial_likelihood.tSZ_PS_Likelihood_FullSkyNPY` |

Theory returns **binned $D_\ell = \ell(\ell+1)C_\ell/(2\pi)$** from hmfast `cl_1h + cl_2h` at the data `ell_eff` values.